In [ ]:
import os
import sys
import base64
import weaviate
from io import BytesIO
from pdf2image import convert_from_path
from groq import Groq
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key = os.getenv("groq_api_key")
if not api_key:
    print("Cannot find API key")
    sys.exit(1)

groq_client = Groq(api_key=api_key)
weaviate_client = weaviate.connect_to_local()
embedding_model = SentenceTransformer('all-mpnet-base-v2') # SentenceTransformer(model) 呼叫時利用 model 轉文字為向量

In [ ]:
def encoding_image(image):
    buffer = BytesIO()
    image.save(buffer, format='JPEG') # 強制轉 JPEG 存入 buffer 的 RAM 空間暫存
    image_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
        # buffer.getvalue() 讀取 buffer 裡面的二進制格式檔案
        # Base64 將原始 JPEG 二進位碼轉成純文字字串, Base64 產出標準 ASCII 字元, 任何作業系統和程式語言都能解讀
        # 將編碼後的 Base64 位元組轉為 UTF-8 字串 (python str), 相容 json
    return image_str

In [ ]:
def image_to_markdown(image_str):
    try:
        completion = groq_client.chat.completions.create(
            # groq_client 呼叫初始化好的user端物件, 包含 API Key
            # .chat.completions 對話補全介面
            # 打包成 JSON 送給 Groq 伺服器
            model = "llama-3.2-90b-vision-preview",
            temperature = 0.1,
            messages = [{"role": "user",
                         "content": [{"type": "text",
                                      "text":(
                                          "You are a professional academic paper parser."
                                          "Convert the content of this image into clean Markdown format following these rules:\n"
                                          "1. Strictly identify the double-column layout and arrange text in the correct reading order.\n"
                                          "2. Preserve the original heading hierarchy (#, ##, ###).\n"
                                          "3. Accurately reconstruct all tables into standard Markdown table format.\n"
                                          "4. Ignore headers, footers, and page numbers.\n"
                                          "5. Output ONLY the Markdown content. Do not include any conversational filler or commentary.")},
                                     {"type": "image_url",
                                      "image_url": {"url": f"data: image/jpeg;base64,{image_str}"}}
                                     ]
                         }]
        )

        paper_context = completion.choices[0].message.content
        return paper_context

    except Exception as e:
            print(f"error: {e}")
            return None